In [13]:
import numpy as np
import torch
import casadi as ca

torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=6, suppress=True)

In [14]:
import numpy as np
import torch
import casadi as ca

torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=6, suppress=True)

In [15]:
# Import your surrogate
from Surrogate_ODE_Model.glc_surrogate_casadi import make_glc_well_surrogate
from Surrogate_ODE_Model.glc_surrogate_torch import glc_surrogate_dx_torch

BSW = 0.20
GOR = 0.05
PI  = 3.0e-6

glc_well = make_glc_well_surrogate(BSW=BSW, GOR=GOR, PI=PI)

# Wrap into a pure CasADi Function for dx only
y_sym = ca.MX.sym("y", 3)
u_sym = ca.MX.sym("u", 2)
dx_sym, out_sym = glc_well(y_sym, u_sym)

F_dx = ca.Function("F_dx", [y_sym, u_sym], [dx_sym], ["y","u"], ["dx"])
print("CasADi dx shape:", F_dx.size_out(0))

CasADi dx shape: (3, 1)


In [16]:
# Pick a "reasonable" point (avoid crazy values first)
y_np = np.array([3e6, 2e6, 8e4], dtype=float)   # [m_G_an, m_G_tb, m_L_tb]
u_np = np.array([0.5, 0.7], dtype=float)        # [u1, u2]

dx_ca = np.array(F_dx(y_np, u_np)).reshape(-1)

y_t  = torch.tensor(y_np)
u_t  = torch.tensor(u_np)
dx_th = glc_surrogate_dx_torch(y_t, u_t, BSW=BSW, GOR=GOR, PI=PI).detach().cpu().numpy().reshape(-1)

print("dx CasADi:", dx_ca)
print("dx Torch :", dx_th)
print("abs err :", np.abs(dx_ca - dx_th))
print("max abs err:", np.max(np.abs(dx_ca - dx_th)))

NameError: name 'kpos' is not defined